# Mask tracking evaluation on DAVIS 2017

This notebook is a small end-to-end test of `evaluate_tracks()` on real video instance masks. It loads 30 annotated frames from the DAVIS 2017 validation split as one video, then verifies perfect mask tracking and a controlled identity switch.

> The first run downloads and caches the DAVIS 2017 train/validation archive. Later runs reuse the local copy.

Run it from an editable checkout with the evaluation and video dependencies installed:

```shell
uv pip install -e . --python .venv/bin/python -C editable_mode=compat
uv pip install trackeval pycocotools imageio-ffmpeg --python .venv/bin/python
```

In [ ]:
from collections import Counter
from pathlib import Path
import sys

import fiftyone as fo
import fiftyone.zoo as foz

print("Python:", sys.executable)
print("FiftyOne:", fo.__file__)

DAVIS = "https://github.com/voxel51/davis-2017"
DATASET_NAME = "tracking-eval-davis-test"

# Make the notebook safe to rerun
if fo.dataset_exists(DATASET_NAME):
    fo.delete_dataset(DATASET_NAME)

_, davis_dir = foz.download_zoo_dataset(DAVIS, split="validation")
Path(davis_dir, "trainval", "Videos").mkdir(exist_ok=True)

dataset = foz.load_zoo_dataset(
    DAVIS,
    split="validation",
    format="video",
    max_samples=30,
    dataset_name=DATASET_NAME,
    download_if_necessary=False,
)
dataset.clone_frame_field("ground_truth", "predictions")

print(dataset)
print(dataset.get_frame_field_schema())

## Perfect tracks

The cloned prediction masks are identical to the ground truth, so HOTA, MOTA, and IDF1 must all be 1.

In [ ]:
perfect = dataset.evaluate_tracks(
    "frames.predictions",
    gt_field="frames.ground_truth",
    method="mots",
    use_masks=True,
    metrics=["HOTA", "CLEAR", "Identity"],
)

perfect.print_report(full_names=True)
assert perfect.hota() == 1.0
assert perfect.mota() == 1.0
assert perfect.idf1() == 1.0
assert perfect.metrics()["IDSW"] == 0

## Inject one identity switch

Split the longest predicted mask track at its midpoint by assigning a fresh track ID to its second half. The masks remain perfect, isolating the identity error.

In [ ]:
sample = dataset.first()
track_counts = Counter(
    detection.index
    for frame in sample.frames.values()
    for detection in (frame.predictions.detections or [])
)
track_id, track_length = track_counts.most_common(1)[0]
track_frames = [
    frame_number
    for frame_number, frame in sample.frames.items()
    if any(
        detection.index == track_id
        for detection in (frame.predictions.detections or [])
    )
]
switch_frame = track_frames[len(track_frames) // 2]
new_id = max(track_counts) + 1

for frame_number, frame in sample.frames.items():
    if frame_number < switch_frame:
        continue

    changed = False
    for detection in frame.predictions.detections or []:
        if detection.index == track_id:
            detection.index = new_id
            changed = True

    if changed:
        frame.save()

print(
    f"Split track {track_id} ({track_length} detections) at frame "
    f"{switch_frame}; new ID: {new_id}"
)

## Evaluate and reload the saved run

The new run should contain exactly one identity switch, lower association scores, persisted results, and per-video sample metrics.

In [ ]:
switched = dataset.evaluate_tracks(
    "frames.predictions",
    gt_field="frames.ground_truth",
    eval_key="davis_mots",
    method="mots",
    use_masks=True,
    metrics=["HOTA", "CLEAR", "Identity"],
)

switched.print_report(full_names=True)
assert switched.metrics()["IDSW"] == 1
assert switched.hota() < perfect.hota()
assert switched.idf1() < perfect.idf1()
assert "davis_mots" in dataset.list_evaluations(type="tracking")

loaded = dataset.load_evaluation_results("davis_mots", cache=False)
assert loaded.metrics() == switched.metrics()
assert dataset.first().davis_mots_idsw == 1

print("Saved per-video HOTA:", dataset.first().davis_mots_hota)
print("Tracking evaluation notebook passed")

## TrackingResults plots

The results object retains TrackEval's HOTA threshold curves. Its identity-switch plot follows the predicted ID assigned to each ground-truth track over time, so switches appear as transitions such as `10 → 20`.

In [ ]:
perfect.compare(switched, values=["HOTA", "DetA", "AssA", "IDF1", "MOTA"])
comparison_plot = perfect.plot_compare(switched)
comparison_plot.show()

hota_plot = switched.plot_hota_curves(sample=sample)
hota_plot.show()

id_switch_plot = switched.plot_id_switches(sample=sample)
id_switch_plot.show()

## Visualize the result

Compare the aggregate scores before and after the identity switch, then show where the predicted track ID changes over time.

In [ ]:
import matplotlib.pyplot as plt

score_names = ["HOTA", "DetA", "AssA", "MOTA", "IDF1"]
positions = range(len(score_names))
perfect_scores = [perfect.metrics()[name] for name in score_names]
switched_scores = [switched.metrics()[name] for name in score_names]

fig, (scores_ax, track_ax) = plt.subplots(1, 2, figsize=(13, 4))

scores_ax.bar(
    [position - 0.2 for position in positions],
    perfect_scores,
    width=0.4,
    label="Perfect",
)
scores_ax.bar(
    [position + 0.2 for position in positions],
    switched_scores,
    width=0.4,
    label="One ID switch",
)
scores_ax.set(
    title="TrackEval scores",
    xticks=list(positions),
    xticklabels=score_names,
    ylim=(0, 1.05),
    ylabel="Score",
)
scores_ax.legend()

predicted_ids = [
    track_id if frame_number < switch_frame else new_id
    for frame_number in track_frames
]
track_ax.plot(
    track_frames,
    [track_id] * len(track_frames),
    label="Ground truth",
)
track_ax.step(
    track_frames,
    predicted_ids,
    where="mid",
    label="Prediction",
)
track_ax.axvline(switch_frame, color="black", linestyle="--", alpha=0.5)
track_ax.set(
    title="Injected identity switch",
    xlabel="Frame",
    ylabel="Track ID",
    yticks=[track_id, new_id],
)
track_ax.legend()

fig.tight_layout()
plt.show()

The dataset is left loaded for inspection in the FiftyOne App. Delete it when finished with `fo.delete_dataset(DATASET_NAME)`.